
# 🚀 Trading Algorítmico en Tiempo Real (Alpaca Streaming)

Versión avanzada tipo hedge fund con:

- Streaming en tiempo real (websockets)
- Strategy engine en vivo
- Risk engine completo
- Execution automática
- Loop continuo


## 1. Setup

In [ ]:

# !pip install alpaca-trade-api websockets pandas numpy

import alpaca_trade_api as tradeapi
import pandas as pd
import numpy as np
import asyncio


## 2. Conexión

In [ ]:

API_KEY = 'YOUR_API_KEY'
API_SECRET = 'YOUR_SECRET'
BASE_URL = 'https://paper-api.alpaca.markets'

api = tradeapi.REST(API_KEY, API_SECRET, BASE_URL, api_version='v2')
account = api.get_account()
print(account.cash)


## 3. Configuración global

In [ ]:

symbol = 'AAPL'
risk_per_trade = 0.01
position = None
prices = []


## 4. Strategy Engine

In [ ]:

def generate_signal(prices):
    if len(prices) < 30:
        return 0
    df = pd.DataFrame(prices, columns=['price'])
    df['ma_fast'] = df['price'].rolling(10).mean()
    df['ma_slow'] = df['price'].rolling(30).mean()
    
    if df['ma_fast'].iloc[-1] > df['ma_slow'].iloc[-1]:
        return 1
    elif df['ma_fast'].iloc[-1] < df['ma_slow'].iloc[-1]:
        return -1
    return 0


## 5. Risk Engine

In [ ]:

def calculate_position_size(price):
    capital = float(api.get_account().cash)
    risk_amount = capital * risk_per_trade
    stop_distance = price * 0.02
    size = int(risk_amount / stop_distance)
    return max(size, 1)


## 6. Execution Engine

In [ ]:

def execute_trade(signal, price):
    global position

    if signal == 1 and position != 'long':
        size = calculate_position_size(price)
        api.submit_order(symbol=symbol, qty=size, side='buy', type='market', time_in_force='gtc')
        position = 'long'
        print('BUY', size)

    elif signal == -1 and position != 'short':
        size = calculate_position_size(price)
        api.submit_order(symbol=symbol, qty=size, side='sell', type='market', time_in_force='gtc')
        position = 'short'
        print('SELL', size)


## 7. Risk Monitor (Drawdown)

In [ ]:

equity_history = []

def check_risk():
    account = api.get_account()
    equity = float(account.equity)
    equity_history.append(equity)
    
    if len(equity_history) > 10:
        peak = max(equity_history)
        drawdown = (peak - equity) / peak
        
        if drawdown > 0.2:
            print('KILL SWITCH ACTIVATED')
            return False
    return True


## 8. Streaming en tiempo real

In [ ]:

async def run_stream():
    conn = tradeapi.Stream(API_KEY, API_SECRET, base_url=BASE_URL)

    @conn.on_bar(symbol)
    async def on_bar(bar):
        price = bar.close
        prices.append(price)

        signal = generate_signal(prices)
        print('Price:', price, 'Signal:', signal)

        if check_risk():
            execute_trade(signal, price)

    await conn.run()


## 9. Ejecutar bot

In [ ]:

await run_stream()
